## 可视化

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from PIL import Image
import matplotlib


def merge_channel(vox):
    vox = vox - vox.min()
    new_voxel = vox[0]
    for i in range(1, vox.shape[0]):
        new_voxel = new_voxel + vox[i]
    return new_voxel


def vis_1_channel(arr, cmap_name="Spectral"):
    cmap = matplotlib.colormaps.get_cmap(cmap_name)
    arr = (arr - arr.min()) / (arr.max() - arr.min()) * 255.0
    arr = arr.astype(np.uint8)

    plt.imshow(arr, cmap=cmap)
    # Turn off axis for better visualization
    plt.axis("off")
    plt.show()


def compare_depth_gt(
    dep,
    gt,
    min_depth,
    max_depth,
    show_colorbar=False,
    set_title=False,
    cmap_name="Spectral",
    pad=10,
    color_bar_width=15,
    dpi=100,
):
    cmap = matplotlib.colormaps.get_cmap(cmap_name)

    # Mask invalid values in the predicted depth
    mask = ~np.isfinite(gt)
    dep[mask] = np.nan

    # Clip depth values to the specified range
    gt = np.clip(gt, min_depth, max_depth)
    dep = np.clip(dep, min_depth, max_depth)

    h, w = dep.shape
    # Compute figure size: Maps tightly fill the space, extra for titles and colorbar
    if show_colorbar:
        # 2 maps + small gap + color bar
        width = 2 * w + 2 * pad + color_bar_width
    else:
        # 2 maps + small gap
        width = 2 * w + pad

    if set_title:
        # Map height + space for titles
        height = h + pad
    else:
        height = h
    figure_width = width / dpi  # 2 maps + small gap
    figure_height = height / dpi  # Map height + space for titles

    fig = plt.figure(figsize=(figure_width, figure_height), dpi=dpi)

    # Predicted Depth Map
    ax1 = fig.add_axes([0, 0, w / width, h / height])
    im1 = ax1.imshow(dep, cmap=cmap, vmin=min_depth, vmax=max_depth)
    ax1.axis("off")

    # Ground Truth Map
    ax2 = fig.add_axes([(w + pad) / width, 0, w / width, h / height])
    im2 = ax2.imshow(gt, cmap=cmap, vmin=min_depth, vmax=max_depth)
    ax2.axis("off")

    # Shared Colorbar
    if show_colorbar:
        colorbar_ax = fig.add_axes(
            [(2 * w + 2 * pad) / width, 0, color_bar_width / width, h / height]
        )  # Adjust height to align with maps
        cbar = fig.colorbar(im1, cax=colorbar_ax, orientation="vertical")
        cbar.set_label("Depth", fontsize=10)

    if set_title:
        ax2.set_title("GT Depth", fontsize=10)
        ax1.set_title("Predicted Depth", fontsize=10)
    plt.show()


def visualize_scene_from_paths(
    img_path,
    vox_path,
    dep_path,
    show_colorbar=False,
    set_title=False,
    cmap_name="Spectral",
    pad=10,
    color_bar_width=15,
    dpi=100,
):
    """
    Visualizes an image, voxel grid, and depth map from file paths.
    """
    cmap = matplotlib.colormaps.get_cmap(cmap_name)

    img = np.array(Image.open(img_path))
    vox = np.load(vox_path)
    dep = np.load(dep_path)

    h, w = dep.shape
    dpi = 100

    # Compute figure size: Maps tightly fill the space, extra for titles and colorbar
    if show_colorbar:
        # 2 maps + small gap + color bar
        width = 3 * w + 3 * pad + color_bar_width
    else:
        # 3 maps + 2 small gaps
        width = 3 * w + 2 * pad

    if set_title:
        # Map height + space for titles
        height = h + pad
    else:
        height = h
    figure_width = width / dpi  # 2 maps + small gap
    figure_height = height / dpi  # Map height + space for titles

    # Plotting
    fig = plt.figure(figsize=(figure_width, figure_height), dpi=dpi)

    # 1. Display the image
    ax1 = fig.add_axes([0, 0, w / width, h / height])
    if img.ndim == 3:  # Color image
        ax1.imshow(img)
    else:  # Grayscale image
        ax1.imshow(img, cmap="gray")
    ax1.axis("off")

    # 2. Display the voxel grid
    vox = merge_channel(vox)
    vox = (vox - vox.min()) / (vox.max() - vox.min()) * 255.0
    vox = vox.astype(np.uint8)

    ax2 = fig.add_axes([(w + pad) / width, 0, w / width, h / height])
    ax2.imshow(vox, cmap="gray")
    ax2.axis("off")

    # 3. Display the depth map
    ax3 = fig.add_axes([(2 * w + 2 * pad) / width, 0, w / width, h / height])
    im3 = ax3.imshow(dep, cmap=cmap)
    ax3.axis("off")
    if show_colorbar:
        colorbar_ax = fig.add_axes(
            [(3 * w + 3 * pad) / width, 0, color_bar_width / width, h / height]
        )  # Adjust height to align with maps
        cbar = fig.colorbar(im3, cax=colorbar_ax, orientation="vertical")
        cbar.set_label("Depth", fontsize=10)

    if set_title:
        ax1.set_title("Image")
        ax2.set_title("Voxel")
        ax3.set_title("Depth Map")

    # plt.tight_layout()
    plt.show()


def plot_spatial_distribution(events):
    # The coordinate origin is shifted to the upper left corner
    events[:, 2] = 0 - events[:, 2]
    positive_events = events[events[:, 3] == 1]
    negative_events = events[events[:, 3] == -1]

    plt.scatter(
        positive_events[:, 1],
        positive_events[:, 2],
        c="red",
        s=1,
        label="Polarity: 1",
        alpha=0.5,
    )
    plt.scatter(
        negative_events[:, 1],
        negative_events[:, 2],
        c="blue",
        s=1,
        label="Polarity: -1",
        alpha=0.5,
    )

    plt.axis("off")
    plt.show()


def plot_3d_events(events):
    # The coordinate origin is shifted to the upper left corner
    events[:, 2] = 0 - events[:, 2]
    events[:, 0] = events[:, 0] - events[0][0]

    fig = plt.figure()
    ax = fig.add_subplot(111, projection="3d")

    ax.scatter(
        events[:, 0],
        events[:, 1],
        events[:, 2],
        c=events[:, 3],
        s=1,
        cmap="bwr",
        alpha=0.7,
    )

    ax.set_xlabel("Time")
    ax.set_ylabel("x")
    ax.set_zlabel("y")
    plt.title("3D Event Visualization")
    plt.show()


def detect_sparsity(events, width=346, height=260):
    corrds = events[:, 1:3]
    unique_coords, counts = np.unique(corrds, axis=0, return_counts=True)
    number_of_unique_coords = len(unique_coords)
    total_positions = width * height
    rate = number_of_unique_coords / total_positions * 100
    print(f"(width x height): {width}x{height} = {total_positions}")
    print(f"Number of coordinates with signals: {number_of_unique_coords}")
    print(f"Proportion: {rate:.2f}%")


def normalize_voxelgrid(event_tensor):
    mask = np.nonzero(event_tensor)
    if mask[0].size > 0:
        mean, stddev = event_tensor[mask].mean(), event_tensor[mask].std()
        print(mean, stddev)
        if stddev > 0:
            event_tensor[mask] = (event_tensor[mask] - mean) / stddev
    return event_tensor


def vis_voxelgrid(voxel):
    new_voxel = merge_channel(voxel)
    vis_1_channel(new_voxel, "gray")


def vis_depth_map(dep, show_colorbar=False, cmap_name="Spectral"):
    cmap = matplotlib.colormaps.get_cmap(cmap_name)

    fig, ax = plt.subplots()
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.axis("off")

    im = ax.imshow(dep, cmap=cmap, extent=[0, dep.shape[1], 0, dep.shape[0]])
    if show_colorbar:
        plt.colorbar(im)

    dpi = fig.get_dpi()
    fig.set_size_inches(dep.shape[1] / dpi, dep.shape[0] / dpi)

    # plt.savefig("test.png")
    plt.show()


def vis_diff(dep, gt):
    diff = gt - dep
    diff = 0 - abs(diff)
    vis_depth_map(diff)


def vis_depth_map_filter(dep, gt, show_colorbar=False, cmap_name="Spectral"):
    cmap = matplotlib.colormaps.get_cmap(cmap_name)

    mask = ~np.isfinite(gt)
    dep[mask] = np.nan

    fig, ax = plt.subplots()
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.axis("off")

    im = ax.imshow(dep, cmap=cmap, extent=[0, dep.shape[1], 0, dep.shape[0]])
    if show_colorbar:
        plt.colorbar(im)

    dpi = fig.get_dpi()
    fig.set_size_inches(dep.shape[1] / dpi, dep.shape[0] / dpi)

    # plt.savefig("test.png")
    plt.show()

#### Vis Scene (image, voxel, GT depth)

In [ ]:
# 1137, 2767, 2917
i = "/data_nvme/sph/mvsec_processed/outdoor_day1/images/02917.png"
v = "/data_nvme/sph/mvsec_processed/outdoor_day1/voxels/02917.npy"
d = "/data_nvme/sph/mvsec_processed/outdoor_day1/depths/02917.npy"
visualize_scene_from_paths(i, v, d, show_colorbar=True, set_title=True)

#### Vis event stream

In [ ]:
p = "/data_nvme/sph/mvsec_processed/outdoor_day1/events/00000.npy"
events = np.load(p)
print(events.shape)
detect_sparsity(events)
# plot_3d_events(events)
plot_spatial_distribution(events)

#### Vis Voxel Grid

In [ ]:
p = "/data_nvme/sph/mvsec_processed/outdoor_day1/voxels/00000.npy"
vox = np.load(p)
vis_voxelgrid(vox)

#### Vis depth map

In [ ]:
# viridis, gray
min_depth, max_depth = 1.97, 80
dep_path = "/home/sph/event/da2-prompt-tuning/results/mvsec_metric_disp_vitl_decoder_day1_20/npy/01137.npy"
gt_path = "/data_nvme/sph/mvsec_processed/outdoor_day1/depths/01137.npy"

dep = np.load(dep_path)
gt = np.load(gt_path)

# vis_depth_map(dep, cmap_name='viridis')
vis_depth_map(dep)

# vis_depth_map_filter(dep, gt, cmap_name='viridis')
# vis_depth_map_filter(dep, gt)

# vis_diff(dep, gt)
# compare_depth_gt(dep, gt, min_depth, max_depth, show_colorbar=True)

In [ ]:
import h5py
import numpy as np

data_hdf5_path = "/data_nvme/sph/mvsec/outdoor_day1_data.hdf5"
gt_hdf5_path = "/data_nvme/sph/mvsec/outdoor_day1_gt.hdf5"

data = h5py.File(data_hdf5_path)
gt = h5py.File(gt_hdf5_path)

In [ ]:
depth_raw = np.array(gt['davis']['left']['depth_image_raw'])
depth_rect = np.array(gt['davis']['left']['depth_image_rect'])
print(depth_raw.shape, depth_rect.shape)

In [ ]:
compare_depth_gt(depth_rect[2917], depth_raw[2917], min_depth=0, max_depth=100, show_colorbar=True, cmap_name="viridis")

In [ ]:
diff = depth_raw[0] - depth_rect[0]
diff = np.clip(diff, -5, 5)
vis_depth_map(diff, show_colorbar=True, cmap_name="viridis")